# SetFit

In [4]:
%pip install setfit sentence_transformers

  Using cached setfit-1.1.3-py3-none-any.whl.metadata (12 kB)
  Using cached evaluate-0.4.6-py3-none-any.whl.metadata (9.5 kB)
Using cached setfit-1.1.3-py3-none-any.whl (75 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 14.6 MB/s eta 0:00:00
Using cached evaluate-0.4.6-py3-none-any.whl (84 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# This version works well (so far)
!pip install "transformers>=4.20.0,<5.0.0" "setfit==1.1.3" -q

# You may have to restart session after installing

zsh:1: /Users/Licas/Desktop/github/NLP/nlpenv/bin/pip: bad interpreter: /Users/Licas/Desktop/NLP/nlpenv/bin/python3.13: no such file or directory

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.13/bin/python3 -m pip install --upgrade pip


In [6]:
from datasets import Dataset
from setfit import SetFitModel, Trainer, TrainingArguments  # ← new API
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
# load and open the dataset from this path '/content/tweets_dataset.csv' and split the dataset into training and evaluation. the training set is randomly selected from 8 text in the dataset.
df = pd.read_csv('15_fake_news_detection(in).csv')
df.head()

In [ ]:
df['label'].value_counts()

In [ ]:
# Rename columns
df = df.rename(columns={'tweet': 'text', 'is_real': 'label'})

# Map True/False in 'label' column to 0/1
df['label'] = df['label'].map({'real': 1, 'fake': 0})

df.head()

In [ ]:
print(df['label'].value_counts(normalize=False))

In [ ]:
# To ensure a balanced proportion of labels, perform stratified sampling for the training set.
# We want 8 texts for training, so we'll aim for 4 of each label if possible.

# Sample 4 texts where label is 0
train_df_label_0 = df[df['label'] == 0].sample(n=4, random_state=42)

# Sample 4 texts where label is 1
train_df_label_1 = df[df['label'] == 1].sample(n=4, random_state=42)

# Concatenate to form the balanced training DataFrame
train_df = pd.concat([train_df_label_0, train_df_label_1])

# Shuffle the training DataFrame to mix the labels
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# The remaining data (for test and eval)
remaining_df = df.drop(train_df.index)

# Split remaining into test and eval (50/50 split)
test_df = remaining_df.sample(frac=0.5, random_state=42)
eval_df = remaining_df.drop(test_df.index)

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))
eval_dataset = Dataset.from_pandas(eval_df.reset_index(drop=True))

# Check sizes
print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}, Eval: {len(eval_dataset)}")
print("Train label distribution:")
print(train_df['label'].value_counts(normalize=False))

In [19]:
# Prepare the evaluation metrics

def compute_metrics(y_pred, y_test):
    accuracy = accuracy_score(y_test, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted"
    )
    return {
        "accuracy":  round(accuracy, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
    }

In [21]:
import os
os.environ["WANDB_DISABLED"] = "true" # If you don't want to use W&B

output_dir = "setfit_2epochs" # Create a folder to store the fined-tuned model

# Load SetFit model from Hub
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-mpnet-base-v2")

# Setting
args = TrainingArguments(
    batch_size=16,
    num_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    metric=compute_metrics,
)

# Train and evaluate
trainer.train()
metrics = trainer.evaluate()
print(">>>> Evaluation Result:", metrics)

# Save the model
trainer.model.save_pretrained(output_dir)

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Map: 100%|██████████| 8/8 [00:00<00:00, 827.83 examples/s]
***** Running training *****
  Num unique pairs = 40
  Batch size = 16
  Num epochs = 2
/Users/Licas/Desktop/github/NLP/nlpenv/lib/python3.13/site-packages

Epoch,Training Loss,Validation Loss
1,0.140300,0.167945
2,0.140300,0.131439


/Users/Licas/Desktop/github/NLP/nlpenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
***** Running evaluation *****


>>>> Evaluation Result: {'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


# **Understanding the Training Arguments:**

`batch_size=16` — The model learns from 16 examples at a time, rather than all at once.

`num_epochs=2` — It reads through the entire training data 2 times to reinforce learning.

`eval_strategy="epoch"` — After each full read-through, test the model on unseen examples to check its progress.

`save_strategy="epoch"` — Take a snapshot of the model after each test.

`load_best_model_at_end=True` — When training finishes, roll back to whichever snapshot performed best — not necessarily the last one.

---

# **Understanding the values produced during the Training process:**
`Epoch` — Which round you're on out of 4 total.

`Training Loss` — How many mistakes the model makes on the data it's learning from. Lower is better. This should drop steadily each epoch.

`Validation Loss` — How many mistakes it makes on data it's never seen before. This is the more honest score — it tells you if the model truly learned or just memorized.

---

### What to watch for

**✅ Healthy training** — Both losses drop together over epochs.

**⚠️ Overfitting** — Training loss keeps dropping but Validation loss starts *rising* (like epochs 3→4 above). The model is memorizing the training data rather than learning general patterns. This is exactly why `load_best_model_at_end=True` exists.

**⚠️ Underfitting** — Both losses are still high even at the last epoch. The model hasn't learned enough — try more epochs or a larger model.

In [22]:
print(metrics)

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


# --> Les métriques sont révélatrices d'un problème sous-jacent: overfitting par data leakage

# Test and use the fine-tuned model

In [ ]:
model = SetFitModel.from_pretrained('setfit_2epochs')

preds = model.predict([
    'I cooked a delicious meal today!',
])
print('Prediction:', np.array(preds)[0])

In [ ]:
preds = model.predict([
    'Some mornings are for sad songs and earl grey tea.',
])
print('Prediction:', np.array(preds)[0])